# 09 — Experiment Comparison & Ensemble Selection

Compare OOF predictions across experiments, identify improvements/regressions,
and determine the best ensemble for submission.

**Experiments tracked:**
| Experiment | Config | Changes | CV AUC |
|---|---|---|---|
| baseline | baseline.yaml | EfficientNet-B0, BCE, basic augs | 0.9475 |
| v2_augment | experiment_v2_augment.yaml | + domain-shift augs | ? |
| v2_augment_focal | experiment_v2_augment_focal.yaml | + focal loss | ? |
| v3_effb1 | experiment_v3_effb1.yaml | + EfficientNet-B1 | ? |

In [ ]:
import os, sys, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score
from pathlib import Path

sys.path.insert(0, os.path.join(os.getcwd(), '..'))
sys.path.insert(0, os.getcwd())

# Try to import from project
try:
    from src.utils.aggregate_oof import find_oof_files
except ImportError:
    pass

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

META_CSV = 'data/raw/train.csv'
EXP_DIR = 'experiments'

## 1 — Load all experiments

In [ ]:
# Define experiments to compare
# Update this dict as you complete experiments
EXPERIMENTS = {
    'baseline': 'experiments/baseline_effb0',
    'v2_augment': 'experiments/v2_augment',
    'v2_augment_focal': 'experiments/v2_augment_focal',
    'v3_effb1': 'experiments/v3_effb1',
}

meta = pd.read_csv(META_CSV)

def load_experiment_oof(base_path, n_folds=5):
    """Load and concatenate OOF predictions from all folds."""
    # Try multiple naming patterns
    patterns = [
        [f'{base_path}_fold{i}/oof_preds.csv' for i in range(n_folds)],
        [f'{base_path}/fold{i}/oof_preds.csv' for i in range(n_folds)],
    ]
    
    for pat in patterns:
        found = [p for p in pat if os.path.exists(p)]
        if len(found) == n_folds:
            dfs = [pd.read_csv(f) for f in found]
            return pd.concat(dfs, ignore_index=True)
    
    # Glob fallback
    parent = str(Path(base_path).parent)
    name = Path(base_path).name
    all_oof = sorted(glob.glob(f'{parent}/{name}*/oof_preds.csv'))
    if all_oof:
        dfs = [pd.read_csv(f) for f in all_oof[:n_folds]]
        return pd.concat(dfs, ignore_index=True)
    
    return None

# Load all available experiments
loaded = {}
for name, path in EXPERIMENTS.items():
    oof = load_experiment_oof(path)
    if oof is not None:
        loaded[name] = oof
        print(f'  ✓ {name}: {len(oof)} samples')
    else:
        print(f'  ✗ {name}: not found')

print(f'\nLoaded {len(loaded)}/{len(EXPERIMENTS)} experiments')

## 2 — Compute macro AUC for each experiment

In [ ]:
def compute_per_species_auc(oof_df, meta_df):
    """Compute per-species AUC from OOF predictions."""
    species_cols = [c for c in oof_df.columns if c != 'filename']
    merged = oof_df[['filename']].merge(
        meta_df[['filename', 'primary_label']], on='filename', how='left')
    
    results = {}
    for sp in species_cols:
        y_true = (merged['primary_label'] == sp).astype(float).values
        y_pred = oof_df[sp].values
        n_pos = y_true.sum()
        if n_pos > 0 and n_pos < len(y_true):
            try:
                results[sp] = roc_auc_score(y_true, y_pred)
            except ValueError:
                results[sp] = None
        else:
            results[sp] = None
    return results

# Compute for all experiments
all_species_auc = {}
summary_rows = []

for name, oof_df in loaded.items():
    species_auc = compute_per_species_auc(oof_df, meta)
    all_species_auc[name] = species_auc
    
    valid = [v for v in species_auc.values() if v is not None]
    macro = np.mean(valid) if valid else 0.0
    n_eval = len(valid)
    n_above_95 = sum(1 for v in valid if v >= 0.95)
    n_below_80 = sum(1 for v in valid if v < 0.80)
    
    summary_rows.append({
        'experiment': name,
        'macro_auc': macro,
        'n_evaluable': n_eval,
        'n_auc_ge_0.95': n_above_95,
        'n_auc_lt_0.80': n_below_80,
        'min_auc': min(valid) if valid else None,
        'median_auc': np.median(valid) if valid else None,
    })

summary = pd.DataFrame(summary_rows).sort_values('macro_auc', ascending=False)
print('\n=== Experiment Summary ===')
print(summary.to_string(index=False))

## 3 — Per-species comparison: baseline vs each experiment

In [ ]:
if 'baseline' in all_species_auc:
    baseline_auc = all_species_auc['baseline']
    
    for exp_name, exp_auc in all_species_auc.items():
        if exp_name == 'baseline':
            continue
        
        print(f'\n{"="*50}')
        print(f'Baseline vs {exp_name}')
        print(f'{"="*50}')
        
        improvements = []
        regressions = []
        
        for sp in baseline_auc:
            b = baseline_auc.get(sp)
            e = exp_auc.get(sp)
            if b is not None and e is not None:
                delta = e - b
                if delta > 0.01:
                    improvements.append((sp, b, e, delta))
                elif delta < -0.01:
                    regressions.append((sp, b, e, delta))
        
        improvements.sort(key=lambda x: x[3], reverse=True)
        regressions.sort(key=lambda x: x[3])
        
        print(f'\nImproved (Δ > +0.01): {len(improvements)} species')
        for sp, b, e, d in improvements[:15]:
            print(f'  {sp:>20s}: {b:.4f} → {e:.4f}  ({d:+.4f})')
        
        print(f'\nRegressed (Δ < -0.01): {len(regressions)} species')
        for sp, b, e, d in regressions[:15]:
            print(f'  {sp:>20s}: {b:.4f} → {e:.4f}  ({d:+.4f})')
else:
    print('Baseline not loaded — cannot compute per-species deltas')

## 4 — Scatter plot: baseline AUC vs experiment AUC per species

In [ ]:
if 'baseline' in all_species_auc and len(loaded) > 1:
    baseline_auc = all_species_auc['baseline']
    other_exps = [k for k in all_species_auc if k != 'baseline']
    
    n_plots = len(other_exps)
    fig, axes = plt.subplots(1, min(n_plots, 3), figsize=(6*min(n_plots, 3), 5))
    if n_plots == 1:
        axes = [axes]
    
    for ax, exp_name in zip(axes, other_exps[:3]):
        exp_auc = all_species_auc[exp_name]
        
        xs, ys = [], []
        for sp in baseline_auc:
            b = baseline_auc.get(sp)
            e = exp_auc.get(sp)
            if b is not None and e is not None:
                xs.append(b)
                ys.append(e)
        
        ax.scatter(xs, ys, alpha=0.5, s=15)
        ax.plot([0, 1], [0, 1], 'r--', alpha=0.5, label='y=x')
        ax.set_xlabel('Baseline AUC')
        ax.set_ylabel(f'{exp_name} AUC')
        ax.set_title(f'Baseline vs {exp_name}')
        ax.set_xlim(0, 1.05)
        ax.set_ylim(0, 1.05)
        ax.legend()
    
    plt.tight_layout()
    plt.show()
else:
    print('Need baseline + at least one other experiment for comparison plot')

## 5 — Ensemble exploration

In [ ]:
if len(loaded) >= 2:
    exp_names = list(loaded.keys())
    
    # Get common filenames across all experiments
    common_files = None
    for name, oof_df in loaded.items():
        files = set(oof_df['filename'].tolist())
        common_files = files if common_files is None else common_files & files
    
    print(f'Common files across all experiments: {len(common_files)}')
    
    # Align all OOF predictions to common files
    aligned = {}
    for name, oof_df in loaded.items():
        df = oof_df[oof_df['filename'].isin(common_files)].sort_values('filename').reset_index(drop=True)
        aligned[name] = df
    
    species_cols = [c for c in aligned[exp_names[0]].columns if c != 'filename']
    filenames = aligned[exp_names[0]]['filename'].values
    
    # Build labels
    merged = pd.DataFrame({'filename': filenames}).merge(
        meta[['filename', 'primary_label']], on='filename', how='left')
    labels = np.zeros((len(filenames), len(species_cols)), dtype=np.float32)
    for i, sp in enumerate(species_cols):
        labels[:, i] = (merged['primary_label'] == sp).astype(float)
    
    def macro_auc(preds):
        aucs = []
        for i in range(labels.shape[1]):
            col = labels[:, i]
            if col.sum() > 0 and col.sum() < len(col):
                try:
                    aucs.append(roc_auc_score(col, preds[:, i]))
                except ValueError:
                    pass
        return np.mean(aucs)
    
    # Try all pairs and the full ensemble
    from itertools import combinations
    
    print(f'\n=== Ensemble Results ===')
    print(f'{"Combination":<50s} {"Macro AUC":>10s}')
    print('-' * 62)
    
    # Individual
    for name in exp_names:
        preds = aligned[name][species_cols].values
        auc = macro_auc(preds)
        print(f'{name:<50s} {auc:>10.4f}')
    
    print()
    
    # Pairs
    for combo in combinations(exp_names, 2):
        preds = np.mean([aligned[n][species_cols].values for n in combo], axis=0)
        auc = macro_auc(preds)
        name = ' + '.join(combo)
        print(f'{name:<50s} {auc:>10.4f}')
    
    # Triples
    if len(exp_names) >= 3:
        print()
        for combo in combinations(exp_names, 3):
            preds = np.mean([aligned[n][species_cols].values for n in combo], axis=0)
            auc = macro_auc(preds)
            name = ' + '.join(combo)
            print(f'{name:<50s} {auc:>10.4f}')
    
    # Full ensemble
    if len(exp_names) >= 4:
        print()
        preds_all = np.mean([aligned[n][species_cols].values for n in exp_names], axis=0)
        auc_all = macro_auc(preds_all)
        print(f'{"ALL":<50s} {auc_all:>10.4f}')
    
    # Weighted ensemble search (simple grid)
    if len(exp_names) == 2:
        print(f'\n=== Weighted Blend: {exp_names[0]} vs {exp_names[1]} ===')
        best_w, best_auc_w = 0.5, 0.0
        for w in np.arange(0.0, 1.05, 0.1):
            preds = w * aligned[exp_names[0]][species_cols].values + \
                    (1 - w) * aligned[exp_names[1]][species_cols].values
            auc = macro_auc(preds)
            if auc > best_auc_w:
                best_w, best_auc_w = w, auc
            print(f'  w={w:.1f}: AUC={auc:.4f}')
        print(f'  Best: w={best_w:.1f}, AUC={best_auc_w:.4f}')
else:
    print('Need at least 2 loaded experiments for ensemble analysis')

## 6 — Post-processing evaluation on best experiment

In [ ]:
# Apply post-processing to the best single experiment
try:
    from src.utils.postprocess import evaluate_postprocessing_grid, clip_predictions
    
    if loaded:
        # Find best experiment
        best_exp = max(loaded.keys(), 
                       key=lambda k: np.mean([v for v in compute_per_species_auc(loaded[k], meta).values() if v is not None]))
        
        print(f'Evaluating post-processing on: {best_exp}')
        
        oof = loaded[best_exp]
        species_cols = [c for c in oof.columns if c != 'filename']
        preds = oof[species_cols].values
        
        merged = oof[['filename']].merge(meta[['filename', 'primary_label']], on='filename', how='left')
        labels = np.zeros((len(oof), len(species_cols)), dtype=np.float32)
        for i, sp in enumerate(species_cols):
            labels[:, i] = (merged['primary_label'] == sp).astype(float)
        
        # Since we have probabilities not logits, convert back
        # logits = log(p / (1-p))
        eps = 1e-7
        preds_clipped = np.clip(preds, eps, 1 - eps)
        logits = np.log(preds_clipped / (1 - preds_clipped))
        
        results = evaluate_postprocessing_grid(labels, logits)
        
        print('\nTop 10 configurations:')
        for r in results['all_results'][:10]:
            delta = r['auc'] - results['baseline_auc']
            print(f"  {r['config']:<30s} AUC={r['auc']:.4f}  ({delta:+.4f})")
except ImportError:
    print('Post-processing module not available — run from project root')

## 7 — Recommendations

In [ ]:
print('\n' + '='*60)
print('RECOMMENDATIONS')
print('='*60)

if summary_rows:
    best = max(summary_rows, key=lambda x: x['macro_auc'])
    baseline_row = next((r for r in summary_rows if r['experiment'] == 'baseline'), None)
    
    print(f'\n1. Best single model: {best["experiment"]} (AUC={best["macro_auc"]:.4f})')
    
    if baseline_row:
        delta = best['macro_auc'] - baseline_row['macro_auc']
        print(f'   Improvement over baseline: {delta:+.4f}')
    
    print(f'\n2. Submit to Kaggle if CV improved. Current LB baseline: 0.799')
    print(f'   Expected LB range: {best["macro_auc"] - 0.15:.3f} to {best["macro_auc"] - 0.10:.3f}')
    print(f'   (assuming 0.10-0.15 CV-LB gap)')
    
    print(f'\n3. Next steps based on results:')
    if delta > 0.005:
        print(f'   ✓ Augmentations helped — keep them for all future experiments')
    else:
        print(f'   ✗ Augmentations did not help much — investigate why')
    
    print(f'\n4. Ensemble diversity: check if different experiments make')
    print(f'   different errors (look at scatter plots above)')
else:
    print('No experiments loaded yet.')